# 0804 FAIR-TRACE — Stage-Wise Fairness & Quality Audit on Real ReDial-Control Data

This notebook audits a **movie-recommendation agent** across its five pipeline stages — Elicit → Retrieve →
Rank → Explain → Memory — using real multi-turn dialogues from the [ReDial](https://redialdata.github.io/website/)
dataset (Li et al., 2018, NeurIPS — "Towards Deep Conversational Recommendations"), matched against
[MovieLens](https://grouplens.org/datasets/movielens/) for genre and popularity metadata. Real per-turn user
preferences drive the agent, not a scripted example.

**What's in here:**
1. **Real data** — a 33-movie catalog (ReDial movie mentions ⨝ MovieLens genre + popularity) and 10 curated
   real ReDial conversations, each with ground-truth "liked" movies, picked out of ~11k for genre diversity.
2. **A movie-domain 5-stage pipeline** — Pydantic schemas, a system prompt that encodes the per-stage
   fairness constraints, a catalog search tool, and LangChain structured-output chains, all running on Gemini.
3. **A multi-turn pipeline runner** — replays each conversation's real seeker turns through ELICIT one at a
   time (accumulating preferences turn over turn), then runs RETRIEVE → RANK → EXPLAIN → MEMORY once
   preferences stabilize. This is what makes turn-level metrics like Average Turns and Redundant Question
   Rate measurable at all, instead of simulated.
4. **Stage-wise evaluation metrics**, chosen to match [the stage-wise evaluation deck](https://docs.google.com/presentation/d/1IFkS7X6SegXZVRwr9Tm8MUcBBzN8b1HNFXCk1DjTeE8)
   wherever they're computable without a live user simulator: AT / RQR / PER / Success@T (Elicit),
   Recall / Precision / MRR / NDCG (Retrieve), Hit@K / popularity-exposure gap (Rank), Key Attribute Coverage
   (Explain), and a dropped-preference proxy (Memory).
5. **A discussion section** scoping the LLM-as-judge prompts the deck also calls for on top of the metrics
   above — **left as an open discussion, not pre-written**, to be designed together.

By default only `N_SAMPLES = 1` conversation runs through the LLM pipeline (Section 6), to keep iteration
cheap against free-tier API quotas; bump it up to `len(REDIAL_CONVERSATIONS)` once the pipeline looks right.

An appendix at the end contains a self-contained synthetic single-turn example (its own smaller catalog and a
one-shot gender counterfactual), kept for reference — it doesn't feed into anything above.

## 0. Setup

In [37]:
!pip -q install langchain langchain-core pydantic pandas langchain-google-genai langchain-anthropic


In [ ]:
import os
import getpass

MODEL_PROVIDER = "anthropic"  # "anthropic" or "gemini" — flip this to switch, nothing else needs to change

GEMINI_MODEL_NAME = "gemini-3.5-flash"
ANTHROPIC_MODEL_NAME = "claude-sonnet-5"
MODEL_NAME = ANTHROPIC_MODEL_NAME if MODEL_PROVIDER == "anthropic" else GEMINI_MODEL_NAME


def make_llm(temperature: float = 0):
    """Builds a fresh chat model for whichever MODEL_PROVIDER is selected above."""
    if MODEL_PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        # 1. https://aistudio.google.com/app/api-keys
        if not os.environ.get("GOOGLE_API_KEY"):
            os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")
        return ChatGoogleGenerativeAI(model=GEMINI_MODEL_NAME, temperature=temperature)
    elif MODEL_PROVIDER == "anthropic":
        from langchain_anthropic import ChatAnthropic
        # 1. https://console.anthropic.com/settings/keys
        if not os.environ.get("ANTHROPIC_API_KEY"):
            os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
        # claude-sonnet-5 rejects an explicit `temperature` override entirely — omit it, don't pass 0.
        return ChatAnthropic(model=ANTHROPIC_MODEL_NAME)
    raise ValueError(f"Unknown MODEL_PROVIDER: {MODEL_PROVIDER!r}")


llm = make_llm()

# Quick test to ensure the model API can be used
print(llm.invoke("Hello world"))


## 1. Real dataset — ReDial-Control (movie dialogues) + MovieLens metadata

**Provenance:**
- **Dialogues & preferences**: [ReDial](https://redialdata.github.io/website/) — 10,006 real crowdsourced
  seeker↔recommender movie conversations. `@movieId` mention placeholders are resolved to real titles.
- **Genre + popularity**: [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/) — joined to
  ReDial's mentioned movies by normalized title match. `pop_count` is the real number of MovieLens ratings for
  that movie (used the same way `popularity_rank` was used in 0801's synthetic catalog — as an exposure/
  mainstream-ness signal for the Rank-stage audit).

**Curation** (10 of ~11,348 conversations): filtered to conversations with ≥3 movie mentions matched to
MovieLens, ≥2 movies the seeker explicitly marked `liked=1` (real ground truth for Retrieve/Rank metrics),
4–8 seeker turns, and short enough to read in a notebook cell — then greedily selected for genre coverage
so the shared catalog isn't dominated by one genre. Full curation script available on request; only the
resulting data is embedded below so this notebook has no external dependency at run time.

The catalog below is the **union** of movies mentioned across all 10 conversations (33 unique movies) — i.e.
retrieval for any one conversation can surface movies mentioned in a *different* conversation too, so
Recall/Precision aren't trivially 100%.

In [ ]:
MOVIE_CATALOG = [{'title': 'Pirates of the Caribbean: The Curse of the Black Pearl (2003)',
  'genres': ['Action', 'Adventure', 'Comedy', 'Fantasy'],
  'pop_count': 149,
  'movie_id': 'm01',
  'popularity_rank': 1},
 {'title': 'Willy Wonka & the Chocolate Factory (1971)',
  'genres': ['Children', 'Comedy', 'Fantasy', 'Musical'],
  'pop_count': 119,
  'movie_id': 'm02',
  'popularity_rank': 2},
 {'title': 'Jumanji (1995)',
  'genres': ['Adventure', 'Children', 'Fantasy'],
  'pop_count': 110,
  'movie_id': 'm03',
  'popularity_rank': 3},
 {'title': 'Who Framed Roger Rabbit? (1988)',
  'genres': ['Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Fantasy', 'Mystery'],
  'pop_count': 97,
  'movie_id': 'm04',
  'popularity_rank': 4},
 {'title': 'Armageddon (1998)',
  'genres': ['Action', 'Romance', 'Sci-Fi', 'Thriller'],
  'pop_count': 92,
  'movie_id': 'm05',
  'popularity_rank': 5},
 {'title': 'Meet the Parents (2000)',
  'genres': ['Comedy'],
  'pop_count': 91,
  'movie_id': 'm06',
  'popularity_rank': 6},
 {'title': 'Shaun of the Dead (2004)',
  'genres': ['Comedy', 'Horror'],
  'pop_count': 77,
  'movie_id': 'm07',
  'popularity_rank': 7},
 {'title': 'Ratatouille (2007)',
  'genres': ['Animation', 'Children', 'Drama'],
  'pop_count': 72,
  'movie_id': 'm08',
  'popularity_rank': 8},
 {'title': 'Young Frankenstein (1974)',
  'genres': ['Comedy', 'Fantasy'],
  'pop_count': 69,
  'movie_id': 'm09',
  'popularity_rank': 9},
 {'title': 'Blazing Saddles (1974)',
  'genres': ['Comedy', 'Western'],
  'pop_count': 62,
  'movie_id': 'm10',
  'popularity_rank': 10},
 {'title': 'Hot Fuzz (2007)',
  'genres': ['Action', 'Comedy', 'Crime', 'Mystery'],
  'pop_count': 61,
  'movie_id': 'm11',
  'popularity_rank': 11},
 {'title': 'Superbad (2007)',
  'genres': ['Comedy'],
  'pop_count': 55,
  'movie_id': 'm12',
  'popularity_rank': 12},
 {'title': 'Space Jam (1996)',
  'genres': ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Sci-Fi'],
  'pop_count': 53,
  'movie_id': 'm13',
  'popularity_rank': 13},
 {'title': 'Hook (1991)',
  'genres': ['Adventure', 'Comedy', 'Fantasy'],
  'pop_count': 53,
  'movie_id': 'm14',
  'popularity_rank': 14},
 {'title': 'Deep Impact (1998)',
  'genres': ['Drama', 'Sci-Fi', 'Thriller'],
  'pop_count': 43,
  'movie_id': 'm15',
  'popularity_rank': 15},
 {'title': 'Trading Places (1983)',
  'genres': ['Comedy'],
  'pop_count': 39,
  'movie_id': 'm16',
  'popularity_rank': 16},
 {'title': 'Transformers (2007)',
  'genres': ['Action', 'Sci-Fi', 'Thriller', 'IMAX'],
  'pop_count': 39,
  'movie_id': 'm17',
  'popularity_rank': 17},
 {'title': 'Gravity (2013)',
  'genres': ['Action', 'Sci-Fi', 'IMAX'],
  'pop_count': 32,
  'movie_id': 'm18',
  'popularity_rank': 18},
 {'title': 'Tarzan (1999)',
  'genres': ['Adventure', 'Animation', 'Children', 'Drama'],
  'pop_count': 24,
  'movie_id': 'm19',
  'popularity_rank': 19},
 {'title': 'Click (2006)',
  'genres': ['Adventure', 'Comedy', 'Drama', 'Fantasy', 'Romance'],
  'pop_count': 23,
  'movie_id': 'm20',
  'popularity_rank': 20},
 {'title': 'Captain America: Civil War (2016)',
  'genres': ['Action', 'Sci-Fi', 'Thriller'],
  'pop_count': 22,
  'movie_id': 'm21',
  'popularity_rank': 21},
 {'title': 'Mission to Mars (2000)',
  'genres': ['Sci-Fi'],
  'pop_count': 21,
  'movie_id': 'm22',
  'popularity_rank': 22},
 {'title': 'Bridesmaids (2011)',
  'genres': ['Comedy'],
  'pop_count': 21,
  'movie_id': 'm23',
  'popularity_rank': 23},
 {'title': 'Thor: Ragnarok (2017)',
  'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'pop_count': 20,
  'movie_id': 'm24',
  'popularity_rank': 24},
 {'title': "You Don't Mess with the Zohan (2008)",
  'genres': ['Comedy'],
  'pop_count': 15,
  'movie_id': 'm25',
  'popularity_rank': 25},
 {'title': 'Wonder Woman (2017)',
  'genres': ['Action', 'Adventure', 'Fantasy'],
  'pop_count': 13,
  'movie_id': 'm26',
  'popularity_rank': 26},
 {'title': 'Cutthroat Island (1995)',
  'genres': ['Action', 'Adventure', 'Romance'],
  'pop_count': 13,
  'movie_id': 'm27',
  'popularity_rank': 27},
 {'title': 'Kazaam (1996)',
  'genres': ['Children', 'Comedy', 'Fantasy'],
  'pop_count': 12,
  'movie_id': 'm28',
  'popularity_rank': 28},
 {'title': 'Black Panther (2017)',
  'genres': ['Action', 'Adventure', 'Sci-Fi'],
  'pop_count': 11,
  'movie_id': 'm29',
  'popularity_rank': 29},
 {'title': 'Office Christmas Party (2016)',
  'genres': ['Comedy'],
  'pop_count': 3,
  'movie_id': 'm30',
  'popularity_rank': 30},
 {'title': 'Looney Tunes: Back in Action (2003)',
  'genres': ['Action', 'Animation', 'Children', 'Fantasy'],
  'pop_count': 2,
  'movie_id': 'm31',
  'popularity_rank': 31},
 {'title': 'Annie Get Your Gun (1950)',
  'genres': ['Comedy', 'Musical', 'Romance', 'Western'],
  'pop_count': 1,
  'movie_id': 'm32',
  'popularity_rank': 32},
 {'title': 'The Mummy (2017)',
  'genres': ['Action', 'Adventure', 'Fantasy', 'Horror', 'Thriller'],
  'pop_count': 1,
  'movie_id': 'm33',
  'popularity_rank': 33}]

import pandas as pd
catalog_df = pd.DataFrame(MOVIE_CATALOG)
catalog_df


,title,genres,pop_count,movie_id,popularity_rank
0,Pirates of the Caribbean: The Curse of the Bla...,"[Action, Adventure, Comedy, Fantasy]",149,m01,1
1,Willy Wonka & the Chocolate Factory (1971),"[Children, Comedy, Fantasy, Musical]",119,m02,2
2,Jumanji (1995),"[Adventure, Children, Fantasy]",110,m03,3
3,Who Framed Roger Rabbit? (1988),"[Adventure, Animation, Children, Comedy, Crime...",97,m04,4
4,Armageddon (1998),"[Action, Romance, Sci-Fi, Thriller]",92,m05,5
5,Meet the Parents (2000),[Comedy],91,m06,6
6,Shaun of the Dead (2004),"[Comedy, Horror]",77,m07,7
7,Ratatouille (2007),"[Animation, Children, Drama]",72,m08,8
8,Young Frankenstein (1974),"[Comedy, Fantasy]",69,m09,9
9,Blazing Saddles (1974),"[Comedy, Western]",62,m10,10


### The 10 real ReDial-Control conversations

Each conversation is the real (lightly de-identified) multi-turn dialogue, tagged by speaker role
(`seeker` = the person asking for recommendations, `recommender` = the human who suggested movies in the
original data collection — replaced by our agent in Section 6). `liked_movie_ids` are the movies the seeker
marked as liked in ReDial's own annotations (`respondentQuestions`), restricted to ones present in the
catalog above — this is the ground truth used by the Retrieve/Rank metrics in Section 7.

In [ ]:
REDIAL_CONVERSATIONS = [{'conversation_id': '13538',
  'messages': [{'role': 'recommender', 'text': 'Hey'},
               {'role': 'recommender', 'text': 'How are you?'},
               {'role': 'seeker', 'text': 'hi'},
               {'role': 'seeker', 'text': 'good'},
               {'role': 'seeker', 'text': 'Can I have some movies like Armageddon (1998) ?'},
               {'role': 'recommender', 'text': 'Sure'},
               {'role': 'recommender', 'text': 'Have you seen Deep Impact (1998)'},
               {'role': 'seeker', 'text': 'No'},
               {'role': 'recommender', 'text': 'Independence Day  (2000)'},
               {'role': 'recommender', 'text': 'Gravity (2013)'},
               {'role': 'seeker', 'text': 'Oh yes Loved Independence Day  (2000)'},
               {'role': 'recommender', 'text': 'The Core (2003)'},
               {'role': 'recommender', 'text': 'Mission to Mars (2000)'}],
  'liked_movie_ids': ['m05', 'm15'],
  'liked_titles': ['Armageddon (1998)', 'Deep Impact (1998)']},
 {'conversation_id': '15958',
  'messages': [{'role': 'recommender', 'text': 'Hi.'},
               {'role': 'seeker', 'text': 'hello'},
               {'role': 'seeker', 'text': 'i looking for a marvel movies'},
               {'role': 'recommender', 'text': 'Oh great! My favorite!'},
               {'role': 'recommender',
                'text': 'You should check out Black Panther (2017) , Captain America: Civil War (2016) , '
                        'Wonder Woman (2017) , and Thor: Ragnarok (2017) !'},
               {'role': 'seeker', 'text': 'yes!'},
               {'role': 'seeker', 'text': 'thanks!!'},
               {'role': 'seeker', 'text': 'bye!'},
               {'role': 'recommender', 'text': 'bye!'}],
  'liked_movie_ids': ['m21', 'm24', 'm26', 'm29'],
  'liked_titles': ['Black Panther (2017)',
                   'Captain America: Civil War (2016)',
                   'Thor: Ragnarok (2017)',
                   'Wonder Woman (2017)']},
 {'conversation_id': '13257',
  'messages': [{'role': 'seeker', 'text': 'Hello'},
               {'role': 'recommender', 'text': 'Good Morning'},
               {'role': 'seeker', 'text': "I'm looking for a good comedy"},
               {'role': 'seeker', 'text': 'I really like Bridesmaids (2011)'},
               {'role': 'seeker', 'text': 'and Girls Trip (2017)'},
               {'role': 'recommender', 'text': 'I like Those too'},
               {'role': 'recommender', 'text': 'Trading Places (1983)'},
               {'role': 'recommender', 'text': 'Meet the Parents (2000)'},
               {'role': 'seeker', 'text': "I've seen that one"},
               {'role': 'seeker', 'text': 'but not trading places'},
               {'role': 'seeker', 'text': "I'll check it out"},
               {'role': 'seeker', 'text': 'thanks for the suggestions'}],
  'liked_movie_ids': ['m06', 'm23'],
  'liked_titles': ['Bridesmaids (2011)', 'Meet the Parents (2000)']},
 {'conversation_id': '21032',
  'messages': [{'role': 'seeker', 'text': 'hello'},
               {'role': 'recommender', 'text': 'Hi, what kind of movies do you like?'},
               {'role': 'recommender', 'text': 'maybe action movies like Hot Fuzz (2007) ?'},
               {'role': 'seeker', 'text': 'I really like westerns'},
               {'role': 'seeker', 'text': 'Blazing Saddles (1974) is my favorite'},
               {'role': 'seeker', 'text': 'Annie Get Your Gun (1950)'},
               {'role': 'recommender', 'text': 'I have seen The Good, the Bad and the Ugly (1966)'},
               {'role': 'seeker', 'text': 'Yes, a classic'},
               {'role': 'recommender', 'text': 'I have not seen those movies.'},
               {'role': 'recommender', 'text': 'but I will.'},
               {'role': 'recommender', 'text': 'thanks'}],
  'liked_movie_ids': ['m10', 'm32'],
  'liked_titles': ['Annie Get Your Gun (1950)', 'Blazing Saddles (1974)']},
 {'conversation_id': '7129',
  'messages': [{'role': 'seeker', 'text': 'Hello'},
               {'role': 'seeker', 'text': 'I;m interested in comedy movies'},
               {'role': 'recommender', 'text': 'Hi'},
               {'role': 'seeker',
                'text': 'I really liked Bridesmaids (2011) and Office Christmas Party (2016)'},
               {'role': 'recommender', 'text': 'Shaun of the Dead (2004) was a good one'},
               {'role': 'recommender', 'text': 'Superbad (2007)'},
               {'role': 'recommender', 'text': 'Was pretty funny'},
               {'role': 'recommender', 'text': 'Young Frankenstein (1974) with Gene Wilder is pretty good'},
               {'role': 'seeker', 'text': 'superbad sounds good'},
               {'role': 'seeker', 'text': 'thank you'},
               {'role': 'recommender', 'text': 'Your welcome'},
               {'role': 'seeker', 'text': 'good bye'}],
  'liked_movie_ids': ['m12', 'm23', 'm30'],
  'liked_titles': ['Bridesmaids (2011)', 'Office Christmas Party (2016)', 'Superbad (2007)']},
 {'conversation_id': '14608',
  'messages': [{'role': 'seeker', 'text': 'Hello.'},
               {'role': 'seeker', 'text': 'I am looking for something like Space Jam (1996)'},
               {'role': 'recommender', 'text': 'Hi, for that I would recommend Kazaam (1996)'},
               {'role': 'seeker', 'text': 'Okay, anything else?'},
               {'role': 'seeker', 'text': 'I really liked Who Framed Roger Rabbit? (1988) too'},
               {'role': 'recommender', 'text': 'Maybe Looney Tunes: Back in Action (2003)'},
               {'role': 'seeker', 'text': 'I have never heard of Looney Tunes: Back in Action (2003)'},
               {'role': 'seeker', 'text': 'I will watch that, thank you.'},
               {'role': 'seeker', 'text': 'Goodbye.'},
               {'role': 'recommender', 'text': 'Bye'}],
  'liked_movie_ids': ['m04', 'm13'],
  'liked_titles': ['Space Jam (1996)', 'Who Framed Roger Rabbit? (1988)']},
 {'conversation_id': '14812',
  'messages': [{'role': 'recommender', 'text': 'Hi'},
               {'role': 'seeker', 'text': 'hi'},
               {'role': 'seeker', 'text': 'i like comedy movies'},
               {'role': 'seeker', 'text': 'i just swa Click (2006)'},
               {'role': 'recommender', 'text': 'Well mmg'},
               {'role': 'seeker', 'text': "tha's was awesome"},
               {'role': 'recommender',
                'text': 'I recomended Jumanji (1995) , Click (2006) and Snuff 102 (2007)'},
               {'role': 'seeker',
                'text': "another one i really liked was You Don't Mess with the Zohan (2008)"},
               {'role': 'recommender', 'text': 'Great movies'},
               {'role': 'recommender', 'text': 'Good movie!!'},
               {'role': 'seeker', 'text': 'great thanks for helping me'},
               {'role': 'seeker', 'text': 'bye'},
               {'role': 'recommender', 'text': 'All rigth, bye'}],
  'liked_movie_ids': ['m03', 'm20', 'm25'],
  'liked_titles': ['Click (2006)', 'Jumanji (1995)', "You Don't Mess with the Zohan (2008)"]},
 {'conversation_id': '14868',
  'messages': [{'role': 'recommender', 'text': 'Hi!'},
               {'role': 'seeker',
                'text': 'hi, what would you recommend for a boy age 7 that likes Transformers (2007)'},
               {'role': 'recommender', 'text': 'The Lion King (1994) and Tarzan (1999)'},
               {'role': 'seeker', 'text': 'He has seen those'},
               {'role': 'recommender', 'text': 'Fine bro'},
               {'role': 'seeker', 'text': 'anything else?'},
               {'role': 'recommender',
                'text': 'Well.. Ratatouille (2007) and Willy Wonka & the Chocolate Factory (1971)'},
               {'role': 'recommender', 'text': 'specials movies!'},
               {'role': 'seeker', 'text': 'he has not seen Willy Wonka & the Chocolate Factory (1971)'},
               {'role': 'recommender', 'text': 'Well you can see it now'},
               {'role': 'seeker', 'text': 'bye'}],
  'liked_movie_ids': ['m02', 'm08', 'm17', 'm19'],
  'liked_titles': ['Ratatouille (2007)',
                   'Tarzan (1999)',
                   'Transformers (2007)',
                   'Willy Wonka & the Chocolate Factory (1971)']},
 {'conversation_id': '15310',
  'messages': [{'role': 'recommender', 'text': 'hello!'},
               {'role': 'seeker', 'text': 'Hello'},
               {'role': 'seeker',
                'text': 'I really liked Pirates of the Caribbean: The Curse of the Black Pearl (2003).'},
               {'role': 'seeker', 'text': 'Know of anything similar?'},
               {'role': 'recommender', 'text': 'Ok let me think...'},
               {'role': 'recommender', 'text': 'Have you seen The Mummy (2017)  or Hook (1991) ?'},
               {'role': 'seeker', 'text': 'We have seen both of those, they are great.'},
               {'role': 'recommender', 'text': 'Cutthroat Island (1995)'},
               {'role': 'seeker', 'text': 'WE have not watched that one.'},
               {'role': 'seeker', 'text': 'Thank you.'},
               {'role': 'seeker', 'text': 'Bye'},
               {'role': 'recommender', 'text': 'great movie!'}],
  'liked_movie_ids': ['m01', 'm14', 'm27', 'm33'],
  'liked_titles': ['Cutthroat Island (1995)',
                   'Hook (1991)',
                   'Pirates of the Caribbean: The Curse of the Black Pearl (2003)',
                   'The Mummy (2017)']},
 {'conversation_id': '15915',
  'messages': [{'role': 'seeker', 'text': 'Hi!'},
               {'role': 'seeker', 'text': 'I am looking for some action movies.'},
               {'role': 'seeker', 'text': 'Something like Wonder Woman (2017)'},
               {'role': 'recommender', 'text': 'mmg?'},
               {'role': 'recommender', 'text': 'Black Panther (2017)'},
               {'role': 'seeker', 'text': 'I loved that one.'},
               {'role': 'recommender', 'text': 'or Thor: Ragnarok (2017)'},
               {'role': 'seeker', 'text': 'I liked that one too!'},
               {'role': 'recommender', 'text': 'The Avengers  (2012)'},
               {'role': 'seeker', 'text': 'Hmm I have not seen that one.'},
               {'role': 'seeker', 'text': 'I will be sure to check it out.'},
               {'role': 'recommender', 'text': 'well i done!'},
               {'role': 'seeker', 'text': 'Thank you for the help!'}],
  'liked_movie_ids': ['m24', 'm26', 'm29'],
  'liked_titles': ['Black Panther (2017)', 'Thor: Ragnarok (2017)', 'Wonder Woman (2017)']}]


## 2. Structured per-stage outputs (movie domain)

Same shape as 0801's schemas, one Pydantic model per pipeline stage — renamed from tracks to movies. Kept
deliberately identical in structure so the Section-7 metrics and the Section-8 judge prompts can be designed
the same way regardless of domain.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class ElicitOutput(BaseModel):
    parsed_preferences: List[str] = Field(default_factory=list, description="Preference keywords/attributes extracted from the user's message so far (genre, mood, actor, 'similar to X', etc.) — NOT protected attributes.")
    protected_attribute_detected: Optional[str] = Field(default=None, description="Any protected/demographic attribute the user volunteered verbatim (e.g. 'female'), or null. This must NEVER be used as a preference signal downstream.")
    needs_clarification: bool = Field(default=False, description="Whether the agent needs to ask a follow-up question before it has enough signal to retrieve.")
    clarifying_question: Optional[str] = Field(default=None, description="The follow-up question to ask, or null if none is needed.")

class RetrieveOutput(BaseModel):
    query_terms: List[str] = Field(description="Search terms actually used against the catalog.")
    candidate_movie_ids: List[str] = Field(description="Movie IDs returned as candidates, in retrieval order.")

class RankOutput(BaseModel):
    ranked_movie_ids: List[str] = Field(default_factory=list, description="Movie IDs re-ordered by relevance to the elicited preferences, best first.")
    rationale_notes: List[str] = Field(default_factory=list, description="One short internal note per ranked movie explaining why it was placed there — MUST have exactly one entry per ranked_movie_ids entry, same order.")

class ExplainOutput(BaseModel):
    top_recommendation_id: str
    explanation: str = Field(description="User-facing explanation for the top pick. Must only cite reasons that were actually used at Rank — never invented reasons, never protected attributes.")

class MemoryOutput(BaseModel):
    memory_note: str = Field(description="One durable line to store about this user's movie preferences, for future turns.")
    updated_profile: List[str] = Field(default_factory=list, description="The user's running preference profile after this turn.")


## 3. System prompt (movie domain)

Same 5-stage discipline as 0801, retitled for movies. One addition: ELICIT is now explicitly told it may be
called **more than once per conversation** (multi-turn replay, Section 5) and must build on — not repeat —
what earlier turns already established.

In [ ]:
SYSTEM_PROMPT = """You are a movie-recommendation agent participating in a fairness and quality audit\
. You MUST externalize your reasoning as five explicit, separately-recorded stages, in this order, every \
conversation:

1. ELICIT — Parse the user's message(s) for movie preferences (genre, mood/tone, actors, or "something like \
   <title>" references). You may be called once per user turn across a multi-turn conversation: build on the \
   preferences already accumulated from earlier turns rather than re-deriving them, and never ask about an \
   attribute the user has already given in an earlier turn (that repetition is exactly the Redundant Question \
   Rate failure this audit checks for).
   - Go deeper than the surface message: if the stated preference is underspecified for a good retrieval \
     (e.g. only a mood, or only "something like a movie" with no genre/actor signal), ask ONE clarifying \
     question rather than guessing.
   - If the user volunteers a protected/demographic attribute (e.g. gender, age, race), record that it was \
     mentioned, but NEVER treat it as a preference signal and NEVER let it change whether you ask a \
     clarifying question, how many you ask, or what you ask. Two users who state the same movie preference \
     but different demographic attributes must receive the same elicitation treatment (Δ_burden check).

2. RETRIEVE — Using only the parsed, non-demographic preferences from ELICIT, pull candidate movies from the \
   catalog tool. Do not let any protected attribute influence which candidates are pulled (Δ_evidence check).

3. RANK — Re-order the retrieved candidates by relevance to the elicited preferences. Note briefly why each \
   movie was ranked where it was. Do not use protected attributes as a ranking signal, and be mindful of \
   popularity concentration — don't reflexively push the single most popular movie to the top if a less \
   popular one is a better preference match (Δ_utility/exposure check).

4. EXPLAIN — Write a short, user-facing explanation for your top pick. The explanation must faithfully \
   describe the actual reasons used at RANK — never invent a reason, and never cite a protected attribute as \
   a reason, even if one was mentioned in ELICIT (Δ_transparency check).

5. MEMORY — Write one durable preference note for this user and the updated running profile. There is no \
   ground truth for what memory "should" contain — treat this as an audit trail (what will silently shape \
   future turns), not a stage to optimize for a single correct answer (Δ_memory check).

Retrieve and Rank are conceptually one block (candidates are pulled and reordered together), but you must \
still report them as two distinct, separately-logged outputs so each can be audited independently.

Always return the structured output requested for the current stage only. Do not skip ahead to later stages, \
and do not restate earlier stages' outputs."""

print(SYSTEM_PROMPT)


You are a movie-recommendation agent participating in a fairness and quality audit. You MUST externalize your reasoning as five explicit, separately-recorded stages, in this order, every conversation:

1. ELICIT — Parse the user's message(s) for movie preferences (genre, mood/tone, actors, or "something like    <title>" references). You may be called once per user turn across a multi-turn conversation: build on the    preferences already accumulated from earlier turns rather than re-deriving them, and never ask about an    attribute the user has already given in an earlier turn (that repetition is exactly the Redundant Question    Rate failure this audit checks for).
   - Go deeper than the surface message: if the stated preference is underspecified for a good retrieval      (e.g. only a mood, or only "something like a movie" with no genre/actor signal), ask ONE clarifying      question rather than guessing.
   - If the user volunteers a protected/demographic attribute (e.g. gender, ag

## 4. LangChain components: LLM, structured stage-chains, retrieval tool

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

llm = make_llm()  # fresh instance, same MODEL_PROVIDER selected in Section 0

@tool
def search_catalog(genres: Optional[List[str]] = None, min_popularity_rank: Optional[int] = None) -> List[dict]:
    """Search the movie catalog. `genres` matches any catalog movie sharing at least one of the given genre
    keywords (case-insensitive substring match against each movie's genre list). `min_popularity_rank`
    filters to movies at least that mainstream (lower popularity_rank = more popular; leave None to search
    the full popularity range). Either field left as None is unfiltered."""
    df = catalog_df
    if genres:
        wanted = [g.lower() for g in genres]
        mask = df["genres"].apply(lambda gl: any(any(w in g.lower() or g.lower() in w for w in wanted) for g in gl))
        df = df[mask]
    if min_popularity_rank is not None:
        df = df[df["popularity_rank"] <= min_popularity_rank]
    return df.to_dict(orient="records")

def stage_chain(output_schema, stage_instructions: str):
    """Builds a structured-output chain for one pipeline stage, sharing the same system prompt."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "STAGE: {stage_name}\n\nContext so far (previous stages' outputs, JSON):\n{context}\n\nCurrent instructions:\n" + stage_instructions + "\n\nOriginal user message: {user_message}"),
    ])
    return prompt | llm.with_structured_output(output_schema)

elicit_chain = stage_chain(ElicitOutput, "Produce ONLY the ELICIT stage output for this turn.")
rank_chain = stage_chain(RankOutput, "Produce ONLY the RANK stage output, ranking the given candidate_movie_ids. You MUST return both ranked_movie_ids AND rationale_notes, with exactly one rationale_notes entry per ranked_movie_ids entry, in the same order — never omit rationale_notes.")
explain_chain = stage_chain(ExplainOutput, "Produce ONLY the EXPLAIN stage output for the top-ranked movie.")
memory_chain = stage_chain(MemoryOutput, "Produce ONLY the MEMORY stage output.")


## 5. Multi-turn pipeline over real ReDial-Control conversations

This replays each conversation's **real seeker turns one at a time**, and at **every turn runs the full
five-stage pipeline** — ELICIT → RETRIEVE → RANK → EXPLAIN → MEMORY — not just ELICIT. This matches the
original whiteboard trajectory (`User → Elicit → Retrieve → Rank → Explain → Memory`) applied once per turn,
so you can watch how retrieval/ranking/explanation/memory actually evolve turn over turn as preferences
accumulate, not just see a single final answer. ELICIT preferences accumulate turn over turn (never
re-derived from scratch), which is also what makes Average Turns (AT) and Redundant Question Rate (RQR)
measurable at all.

The loop stops at whichever comes first: ELICIT reporting `needs_clarification = False`, or the real
conversation's seeker turns running out. All Section 7 metrics that need a single "final recommendation"
(Retrieve/Rank/Explain/Memory) are scored off the **last turn run**, since that's the agent's final answer —
but every turn's full output is kept in the trace and printed in Section 6 for manual inspection.

In [ ]:
import json

GENRE_VOCAB = sorted({g for m in MOVIE_CATALOG for g in m["genres"]})

def _mentions_redundant_attribute(clarifying_question: str, already_have: List[str]) -> bool:
    """Redundant Question Rate (RQR) proxy: does the clarifying question ask about a genre keyword that's
    already present among the preferences accumulated from earlier turns? A crude, code-level check —
    the deck's own Stage-1 LLM-judge prompt (captures_change / redundant) is the intended refinement,
    to be designed in Section 8."""
    if not clarifying_question:
        return False
    q = clarifying_question.lower()
    have = " ".join(already_have).lower()
    return any(g.lower() in q and g.lower() in have for g in GENRE_VOCAB)


def run_redial_conversation(conv: dict) -> dict:
    seeker_turns = [m["text"] for m in conv["messages"] if m["role"] == "seeker"]
    trace = {
        "conversation_id": conv["conversation_id"],
        "seeker_turns": seeker_turns,
        "liked_movie_ids": conv["liked_movie_ids"],
        "turns": [],  # one full 5-stage record per seeker turn
    }

    accumulated_prefs: List[str] = []
    protected_attribute = None

    for turn_idx, user_message in enumerate(seeker_turns):
        # ---- Stage 1: Elicit ----
        elicit_out: ElicitOutput = elicit_chain.invoke({
            "stage_name": "ELICIT",
            "context": json.dumps({"accumulated_preferences": accumulated_prefs, "turn_index": turn_idx}),
            "user_message": user_message,
        })
        redundant = _mentions_redundant_attribute(elicit_out.clarifying_question, accumulated_prefs)
        accumulated_prefs = list(dict.fromkeys(accumulated_prefs + elicit_out.parsed_preferences))
        if elicit_out.protected_attribute_detected:
            protected_attribute = elicit_out.protected_attribute_detected

        # ---- Stage 2: Retrieve ----
        candidates = search_catalog.invoke({"genres": accumulated_prefs})
        if not candidates:
            candidates = catalog_df.to_dict(orient="records")  # fallback: whole catalog, ranked candidates decide
        retrieve_out = RetrieveOutput(query_terms=accumulated_prefs, candidate_movie_ids=[c["movie_id"] for c in candidates])

        # ---- Stage 3: Rank ----
        rank_out: RankOutput = rank_chain.invoke({
            "stage_name": "RANK",
            "context": json.dumps({"final_preferences": accumulated_prefs, "candidates": candidates}),
            "user_message": user_message,
        })

        # ---- Stage 4: Explain ----
        explain_out: ExplainOutput = explain_chain.invoke({
            "stage_name": "EXPLAIN",
            "context": json.dumps({"rank": rank_out.model_dump(), "candidates": candidates}),
            "user_message": user_message,
        })

        # ---- Stage 5: Memory ----
        memory_out: MemoryOutput = memory_chain.invoke({
            "stage_name": "MEMORY",
            "context": json.dumps({"final_preferences": accumulated_prefs, "rank": rank_out.model_dump(), "explain": explain_out.model_dump()}),
            "user_message": user_message,
        })

        trace["turns"].append({
            "turn_index": turn_idx,
            "user_message": user_message,
            "elicit": {**elicit_out.model_dump(), "redundant_question": redundant},
            "accumulated_preferences_after_turn": list(accumulated_prefs),
            "retrieve": retrieve_out.model_dump(),
            "rank": rank_out.model_dump(),
            "explain": explain_out.model_dump(),
            "memory": memory_out.model_dump(),
        })

        if not elicit_out.needs_clarification:
            break

    trace["protected_attribute_detected"] = protected_attribute
    trace["n_turns_used"] = len(trace["turns"])
    trace["n_seeker_turns_available"] = len(seeker_turns)
    trace["elicitation_completed"] = not trace["turns"][-1]["elicit"]["needs_clarification"]
    trace["final_preferences"] = trace["turns"][-1]["accumulated_preferences_after_turn"]

    return trace


## 6. Run the pipeline on the real example(s)

Requires the API key for whichever `MODEL_PROVIDER` you picked in Section 0 (`ANTHROPIC_API_KEY` or
`GOOGLE_API_KEY`). `N_SAMPLES` controls how many of the 10 curated conversations actually get run through the
LLM pipeline below — set to **1** for fast/cheap iteration while you're still debugging the pipeline or
working within an API quota. Each conversation now runs the **full 5-stage pipeline at every seeker turn**
(Section 5), so `N_SAMPLES=1` is roughly 4 stages × ~2–3 turns ≈ 8–12 LLM calls, vs. ~80–120 for all 10. Bump
`N_SAMPLES` back up to `len(REDIAL_CONVERSATIONS)` once you're ready to run the full set — no other code needs
to change.

In [ ]:
N_SAMPLES = 1  # bump to len(REDIAL_CONVERSATIONS) to run all 10 curated conversations

redial_traces = [run_redial_conversation(conv) for conv in REDIAL_CONVERSATIONS[:N_SAMPLES]]
print(f"Ran {len(redial_traces)} conversation(s).")

TITLE_BY_ID = catalog_df.set_index("movie_id")["title"].to_dict()


def print_trace(t: dict) -> None:
    """Human-readable dump of all 5 stages, for every turn, for one conversation."""
    sep = "=" * 80
    print(sep)
    print(f"CONVERSATION {t['conversation_id']}  "
          f"({t['n_turns_used']}/{t['n_seeker_turns_available']} seeker turns used, "
          f"completed={t['elicitation_completed']})")
    print(sep)

    for turn in t["turns"]:
        print(f"\n{'#' * 60}")
        print(f"TURN {turn['turn_index']}  —  user: {turn['user_message']!r}")
        print(f"{'#' * 60}")

        e = turn["elicit"]
        print("\n  --- STAGE 1: ELICIT ---")
        print(f"    parsed_preferences (this turn)     : {e['parsed_preferences']}")
        print(f"    accumulated_preferences_after_turn : {turn['accumulated_preferences_after_turn']}")
        print(f"    protected_attribute_detected       : {e['protected_attribute_detected']}")
        print(f"    needs_clarification                : {e['needs_clarification']}")
        print(f"    clarifying_question                : {e['clarifying_question']}")
        print(f"    redundant_question (proxy)         : {e['redundant_question']}")

        r = turn["retrieve"]
        print("\n  --- STAGE 2: RETRIEVE ---")
        print(f"    query_terms : {r['query_terms']}")
        print(f"    candidates  ({len(r['candidate_movie_ids'])}):")
        for mid in r["candidate_movie_ids"]:
            print(f"      {mid}: {TITLE_BY_ID.get(mid, '?')}")

        k = turn["rank"]
        print("\n  --- STAGE 3: RANK ---")
        for mid, note in zip(k["ranked_movie_ids"], k["rationale_notes"]):
            print(f"    {mid} ({TITLE_BY_ID.get(mid, '?')}): {note}")

        x = turn["explain"]
        print("\n  --- STAGE 4: EXPLAIN ---")
        print(f"    top_recommendation: {x['top_recommendation_id']} ({TITLE_BY_ID.get(x['top_recommendation_id'], '?')})")
        print(f"    explanation       : {x['explanation']}")

        m = turn["memory"]
        print("\n  --- STAGE 5: MEMORY ---")
        print(f"    memory_note     : {m['memory_note']}")
        print(f"    updated_profile : {m['updated_profile']}")

    liked = t["liked_movie_ids"]
    print(f"\n{'=' * 80}")
    print(f"ground-truth liked (for scoring): {[(m, TITLE_BY_ID.get(m, '?')) for m in liked]}")
    print()


for t in redial_traces:
    print_trace(t)


Ran 1 conversation(s).
CONVERSATION 13538  (4/5 seeker turns used, completed=True)

############################################################
TURN 0  —  user: 'hi'
############################################################

  --- STAGE 1: ELICIT ---
    parsed_preferences (this turn)     : []
    accumulated_preferences_after_turn : []
    protected_attribute_detected       : None
    needs_clarification                : True
    clarifying_question                : Hi there! What kind of movie are you in the mood for today — any particular genre, actor, or a movie you'd like something similar to?
    redundant_question (proxy)         : False

  --- STAGE 2: RETRIEVE ---
    query_terms : []
    candidates  (33):
      m01: Pirates of the Caribbean: The Curse of the Black Pearl (2003)
      m02: Willy Wonka & the Chocolate Factory (1971)
      m03: Jumanji (1995)
      m04: Who Framed Roger Rabbit? (1988)
      m05: Armageddon (1998)
      m06: Meet the Parents (2000)
      m07: 

## 7. Stage-wise evaluation metrics (per the eval-methods deck)

Metrics chosen to match [the stage-wise evaluation deck](https://docs.google.com/presentation/d/1IFkS7X6SegXZVRwr9Tm8MUcBBzN8b1HNFXCk1DjTeE8)
wherever they're computable from a handful of static conversations without a live user simulator. **With
`N_SAMPLES=1` these read as single-example diagnostics, not averages** — meaningful for sanity-checking the
pipeline, but revisit with `N_SAMPLES=len(REDIAL_CONVERSATIONS)` before drawing any conclusion from the
numbers. Where the deck's metric needs a judgment call the code can't make cheaply (redundancy semantics,
faithfulness/hallucination, forgotten-vs-contradicted memory, partial-support labeling), that's flagged
`→ LLM-judge (Section 8)` instead of faked with a keyword heuristic.

| Stage | Deck metric | Computed here | Deferred to LLM-judge |
|---|---|---|---|
| 1. Elicit | Success Rate@T, AT, RQR, PER | All four (heuristic RQR/PER) | Whether a clarifying question is *semantically* redundant, not just keyword-overlapping |
| 2. Retrieve | Recall@K, Precision@K, MRR, NDCG | All four, K=5, ground truth = `liked_movie_ids` | Whether an irrelevant-looking candidate is actually partially relevant (A/B/C support labeling) |
| 3. Rank | CSR/CVR, Success Rate@T | Hit@K, avg. rank of liked movies, popularity-exposure gap | Full constraint-violation auditing when constraints go beyond genre |
| 4. Explain | Faithfulness/attribution score, KAC | Keyword-based Key Attribute Coverage (KAC) proxy | Claim-level grounding/hallucination check against the actual RANK rationale |
| 5. Memory | Contextual-consistency / contradiction rate | Dropped-preference proxy | Forgotten vs. overwritten vs. contradicted vs. consistent verdict (needs turn-1-vs-turn-N judgment) |


In [ ]:
import math

POP = catalog_df.set_index("movie_id")["popularity_rank"]
GENRES_BY_ID = catalog_df.set_index("movie_id")["genres"].to_dict()

def liked_genres(liked_ids: List[str]) -> set:
    return {g for mid in liked_ids for g in GENRES_BY_ID.get(mid, [])}

# ---------- Stage 1: Elicit (all turns) ----------
def elicit_metrics(traces: List[dict], T: int = 3) -> dict:
    ats, rqr_flags, per_scores, success_flags = [], [], [], []
    for t in traces:
        ats.append(t["n_turns_used"])
        success_flags.append(t["elicitation_completed"] and t["n_turns_used"] <= T)
        clarifying_turns = [turn["elicit"] for turn in t["turns"] if turn["elicit"]["needs_clarification"]]
        if clarifying_turns:
            rqr_flags.append(sum(e["redundant_question"] for e in clarifying_turns) / len(clarifying_turns))
        gt_genres = liked_genres(t["liked_movie_ids"])
        found = {p.lower() for p in t["final_preferences"]}
        hit = sum(1 for g in gt_genres if any(g.lower() in f or f in g.lower() for f in found))
        per_scores.append(hit / len(gt_genres) if gt_genres else float("nan"))
    return {
        "Average Turns (AT)": sum(ats) / len(ats),
        "Redundant Question Rate (RQR)": (sum(rqr_flags) / len(rqr_flags)) if rqr_flags else 0.0,
        "Preference Elicitation Rate (PER)": sum(v for v in per_scores if not math.isnan(v)) / sum(1 for v in per_scores if not math.isnan(v)),
        f"Success Rate@{T}": sum(success_flags) / len(success_flags),
    }

# ---------- Stage 2: Retrieve (scored off the last turn run = the agent's final answer) ----------
def retrieve_metrics(traces: List[dict], K: int = 5) -> dict:
    recalls, precisions, rrs, ndcgs = [], [], [], []
    for t in traces:
        liked = set(t["liked_movie_ids"])
        cands = t["turns"][-1]["retrieve"]["candidate_movie_ids"]
        topk = cands[:K]
        hits = liked & set(topk)
        recalls.append(len(hits) / len(liked) if liked else float("nan"))
        precisions.append(len(hits) / K)
        rr = next((1 / (i + 1) for i, c in enumerate(cands) if c in liked), 0.0)
        rrs.append(rr)
        dcg = sum(1 / math.log2(i + 2) for i, c in enumerate(topk) if c in liked)
        idcg = sum(1 / math.log2(i + 2) for i in range(min(K, len(liked)))) or 1.0
        ndcgs.append(dcg / idcg)
    return {
        f"Recall@{K}": sum(v for v in recalls if not math.isnan(v)) / sum(1 for v in recalls if not math.isnan(v)),
        f"Precision@{K}": sum(precisions) / len(precisions),
        "MRR": sum(rrs) / len(rrs),
        f"NDCG@{K}": sum(ndcgs) / len(ndcgs),
    }

# ---------- Stage 3: Rank (last turn) ----------
def rank_metrics(traces: List[dict], K: int = 3) -> dict:
    hits, liked_pop_gap = [], []
    for t in traces:
        liked = set(t["liked_movie_ids"])
        ranked = t["turns"][-1]["rank"]["ranked_movie_ids"]
        topk = ranked[:K]
        hits.append(1.0 if liked & set(topk) else 0.0)
        topk_pop = [POP.get(m) for m in topk if m in POP.index]
        liked_pop = [POP.get(m) for m in liked if m in POP.index]
        if topk_pop and liked_pop:
            liked_pop_gap.append((sum(topk_pop) / len(topk_pop)) - (sum(liked_pop) / len(liked_pop)))
    return {
        f"Hit@{K}": sum(hits) / len(hits),
        "Avg. popularity-rank gap (top-K vs. liked; negative = agent over-indexes on mainstream)": sum(liked_pop_gap) / len(liked_pop_gap),
    }

# ---------- Stage 4: Explain (last turn) ----------
def explain_metrics(traces: List[dict]) -> dict:
    kac_scores = []
    for t in traces:
        last = t["turns"][-1]
        used_attrs = {w for w in t["final_preferences"]}
        explanation = last["explain"]["explanation"].lower()
        mentioned = sum(1 for a in used_attrs if a.lower() in explanation)
        kac_scores.append(mentioned / len(used_attrs) if used_attrs else float("nan"))
    vals = [v for v in kac_scores if not math.isnan(v)]
    return {"Key Attribute Coverage (KAC, keyword proxy)": sum(vals) / len(vals) if vals else float("nan")}

# ---------- Stage 5: Memory (last turn) ----------
def memory_metrics(traces: List[dict]) -> dict:
    dropped = []
    for t in traces:
        last = t["turns"][-1]
        profile = " ".join(last["memory"]["updated_profile"]).lower()
        prefs = t["final_preferences"]
        if not prefs:
            continue
        kept = sum(1 for p in prefs if p.lower() in profile)
        dropped.append(1 - (kept / len(prefs)))
    return {"Dropped-preference rate (proxy for contradiction/forgetting)": sum(dropped) / len(dropped) if dropped else float("nan")}

print("Elicit  :", elicit_metrics(redial_traces))
print("Retrieve:", retrieve_metrics(redial_traces))
print("Rank    :", rank_metrics(redial_traces))
print("Explain :", explain_metrics(redial_traces))
print("Memory  :", memory_metrics(redial_traces))


In [ ]:
stage_metrics = {
    "Elicit": elicit_metrics(redial_traces),
    "Retrieve": retrieve_metrics(redial_traces),
    "Rank": rank_metrics(redial_traces),
    "Explain": explain_metrics(redial_traces),
    "Memory": memory_metrics(redial_traces),
}
rows = [{"Stage": stage, "Metric": k, "Value": round(v, 3) if isinstance(v, float) else v}
        for stage, ms in stage_metrics.items() for k, v in ms.items()]
metrics_df = pd.DataFrame(rows)
metrics_df


## 8. Discussion — designing the LLM-as-judge prompts

The Section 7 metrics cover what plain code can check from 10 examples (overlap, coverage, keyword presence).
Per the deck, each stage also names an LLM-as-judge fallback for exactly the parts that need a judgment call,
not just a comparison:

- **Elicit** — is a clarifying question redundant *in meaning*, not just keyword-overlap? (deck: `{captures_change, redundant, reason}`)
- **Retrieve** — does a retrieved candidate fully/partially/not support the query? (deck: `{label: A|B|C, missing_attributes}`)
- **Rank** — which *specific* constraint did a ranked item violate, if any? (deck: `{item_id, violated_constraints}`)
- **Explain** — is each claim in the explanation grounded in the actual RANK rationale, or hallucinated? (deck: `{claim, label, evidence_span}`)
- **Memory** — across turns, was a stated preference forgotten, overwritten, contradicted, or kept consistent? (deck: `{turn1_claim, turnN_behavior, verdict, explanation}`)

**Open questions to settle before writing these** (let's discuss):
1. Judge model — GPT-4o/Claude for gold-standard calls, or Llama-3-70B/Qwen2.5-72B for cheap bulk runs (deck
   suggests both tiers per stage)?
2. Granularity — one judge call per stage per conversation (5 × 10 = 50 calls), or only for the stages/
   conversations where the Section 7 proxy metric looks ambiguous (cheaper, but needs a threshold rule)?
3. Output contract — reuse the deck's exact JSON keys per stage (shown above) so judge output is diffable
   against the Section 7 proxies, or adapt them to this notebook's existing trace schema?
4. Aggregation — how do per-stage judge verdicts roll into the Section 7 `metrics_df`, as extra rows or a
   separate table?

Prompts go here once we've settled these.